# 05 — Prediction market → TradFi lead-lag research

**Objective:** test whether prediction-market prices on macro events (Fed, CPI,
geopolitical) lead TradFi instruments (TLT, SPY, VIX proxy, DXY).

**Output:** lag at which PM-to-TradFi correlation peaks. If lead < 15 minutes or
|correlation| < 0.30, the strategy returns zero weight.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.kalshi_client import KalshiClient
from src.data.polymarket_client import PolymarketClient
from src.strategies.prediction_market_signals import LeadLagResearchStrategy, LeadLagConfig

k = KalshiClient()
p = PolymarketClient()
print('Kalshi available:', k.available())
print('Polymarket available:', p.available())

### Browse Kalshi markets (Fed / CPI focus)

Kalshi public market data needs no auth.

In [ ]:
km = k.list_markets(status='open', limit=200)
if not km.empty:
    print(f'{len(km)} open markets, columns: {km.columns.tolist()[:10]}')
    if 'title' in km.columns:
        fed = km[km['title'].str.contains('Fed|FOMC|rate', case=False, na=False)]
        # Show whichever price columns exist (Kalshi schema varies)
        wanted = [c for c in ['ticker','title','yes_bid','yes_ask','last_price','volume_24h','volume'] if c in fed.columns]
        print(f'{len(fed)} Fed-related markets:')
        fed[wanted].head(15)
else:
    print('No markets returned')

### Browse Polymarket

Polymarket is permissionless; their Gamma API is open.

In [ ]:
pm = p.list_markets(active=True, limit=200)
if not pm.empty:
    print(f'{len(pm)} active markets, columns: {pm.columns.tolist()[:10]}')
    pm.head()

### Lead-lag analysis (skeleton)

For a real study you need:
1. minute-by-minute PM probability series for an event (download from `get_market_history` on Kalshi or `get_price_history` on Polymarket).
2. minute-by-minute TradFi price for the comparison instrument (Polygon or Alpaca).

Below is a synthetic example showing how the strategy decides whether the signal is tradeable. Replace with real data once your minute-level data feed is up.

In [ ]:
np.random.seed(0)
idx = pd.date_range('2026-01-01', periods=2000, freq='1min', tz='UTC')
pm_prob = pd.Series(np.cumsum(np.random.normal(0,0.001,2000))+0.5, index=idx).clip(0.01,0.99)
tlt = pm_prob.shift(30) * 0.5 + np.random.normal(0, 0.002, 2000).cumsum() + 100
tlt = tlt.bfill()
df = pd.DataFrame({'FED-RATE-DEC': pm_prob, 'TLT': tlt})

strat = LeadLagResearchStrategy(LeadLagConfig(pm_symbol='FED-RATE-DEC', tradfi_symbol='TLT'))
_ = strat.signal(df)
print('lead-lag analysis:', strat.last_analysis)

### Production recipe

Run nightly:
1. For each upcoming Fed/CPI event window, pull Kalshi + Polymarket minute data.
2. Align to SOFR futures (or TLT proxy) minute bars from Polygon.
3. Run `LeadLagResearchStrategy.analyze_leadlag()`; persist `(event_id, lead_min, corr)`.
4. Trade only when |lead| > 15 min AND |corr| > 0.30 AND survives a 6-month rolling OOS check.

**Anti-bias rule:** PM prices used in the lookback window must have timestamps strictly before the TradFi window being predicted.